# AQUA20 — Marine Species Classification under Different Data-Augmentation Policies

Two-stage transfer learning on AQUA20 (20-class marine species), studying how the training-image
**data augmentation** pipeline changes ResNet50's performance.

Each policy in `AUGMENTATIONS` (`current` / `strong` / `rare`) is trained, evaluated and explained in
the *same* Kaggle run. Every policy gets its own artifacts:

    analysis/<policy>/summary.json, per_class.csv, epoch_log.csv, ...
    analysis/augmentation_comparison.csv   analysis/augmentation_comparison.png
    confusion_matrix_resnet50_<policy>.png
    gradcam/resnet50_<policy>/
    training_state_resnet50_<policy>.json
    weights_resnet50_<policy>.pth

Model: `resnet50`


In [ ]:
!pip install -q "torch>=2.0.0" "torchvision>=0.15.0" "datasets>=2.14.0" "scikit-learn>=1.2.0" "matplotlib>=3.7.0" "seaborn>=0.12.0" "tqdm>=4.65.0" "numpy>=1.24.0" "Pillow>=9.5.0" "opencv-python-headless>=4.7.0"


In [ ]:
import os, json, random, time
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.models import (
    ResNet50_Weights, ConvNeXt_Tiny_Weights, ConvNeXt_Base_Weights, Swin_V2_B_Weights,
)
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from datasets import load_dataset, load_dataset_builder
from collections import Counter
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

matplotlib.use("Agg")


## Configuration

In [ ]:
# Architecture: ResNet50, benchmarked under different data-augmentation policies.
MODEL_NAME = "resnet50"

NUM_CLASSES = 20
BATCH_SIZE = 32
EPOCHS_STAGE1 = 25
EPOCHS_STAGE2 = 75
LR_HEAD = 1e-3
LR_BACKBONE = 1e-5
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = os.cpu_count()

# Which validation metric picks the best checkpoint: "macro_f1" | "accuracy".
# Training is class-balanced (WeightedRandomSampler) and macro-F1 is the
# headline number, so accuracy — dominated by fish + coral (~54% of val) — is not.
SELECTION_METRIC = "macro_f1"

# ---- Data-augmentation policies to run, in order ---------------------------
#   current : the baseline pipeline used by v2/v4 (the ResNet50 comparison point)
#             = RandomResizedCrop + HorizontalFlip + mild ColorJitter
#   strong  : aggressive policy targeting the AQUA20 challenges
#             = smaller crop scale (0.4) + rotation 30 + strong colour jitter
#               + GaussianBlur (turbidity) + RandomErasing (occlusion)
#   rare    : baseline PLUS a per-class policy — classes with fewer than
#             RARE_THRESHOLD training samples get an extra random distortion with
#             probability RARE_EXTRA_PROB, on top of the WeightedRandomSampler
#             that already over-samples them. Only rare-class samples are affected.
AUGMENTATIONS = ["current", "strong", "rare"]

RARE_THRESHOLD = 80      # classes with fewer train samples are "rare"
RARE_EXTRA_PROB = 0.5    # probability a rare-class sample also gets the extra distortion

# Per-policy wall-clock budget on Stage 2. A full ResNet50 run is ~1.7 h, so 3
# policies are ~5 h. Each policy gets a 2.5 h cap, keeping the whole session
# under the ~9 h limit. Hitting a cap keeps the best checkpoint so far and moves
# on — a session killed at the limit saves NOTHING.
TIME_BUDGET_SEC = 2.5 * 3600

# Reproducibility. Fixes weight init, data order, sampler draws and augmentation.
# NOT bit-exact: cuDNN autotuning and some GPU kernels stay nondeterministic.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ---- Per-policy artifact names ----------------------------------------------
# `kaggle kernels output` downloads alphabetically and truncates on big payloads,
# so names put the small important files (analysis/...) first and the checkpoint
# last. Every name is policy-scoped, so merging all three policies is collision-free.
def variant_paths(augment):
    return {
        "analysis":    f"analysis/{augment}",
        "gradcam":     f"gradcam/resnet50_{augment}",
        "confusion":   f"confusion_matrix_resnet50_{augment}.png",
        "state":       f"training_state_resnet50_{augment}.json",
        "checkpoint":  f"weights_resnet50_{augment}.pth",
        "epoch_log":   f"analysis/{augment}/epoch_log.csv",
    }

for augment in AUGMENTATIONS:
    os.makedirs(variant_paths(augment)["analysis"], exist_ok=True)
    os.makedirs(variant_paths(augment)["gradcam"], exist_ok=True)
os.makedirs("analysis", exist_ok=True)

print(f"Device: {DEVICE} | Model: {MODEL_NAME}")
print(f"Augmentation policies to run: {AUGMENTATIONS}")
print(f"Selection metric: {SELECTION_METRIC} | Seed: {SEED}")
print(f"Rare threshold: {RARE_THRESHOLD} | Rare extra prob: {RARE_EXTRA_PROB}")
print(f"Epochs: {EPOCHS_STAGE1} (head) + {EPOCHS_STAGE2} (fine-tune) "
      f"(cap {TIME_BUDGET_SEC/3600:.1f} h per policy)")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB), "
          f"{torch.cuda.device_count()} visible")


## Data Module — per-policy transforms + WeightedRandomSampler

In [ ]:
def train_transforms(augment, is_rare=False):
    if augment == "current":
        tile = [
            transforms.RandomResizedCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        ]
    elif augment == "strong":
        tile = [
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.2),
            transforms.GaussianBlur(5, sigma=(0.1, 2.0)),   # turbidity
        ]
    elif augment == "rare":
        tile = [
            transforms.RandomResizedCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        ]
        if is_rare:
            # Per-class policy: rare classes also get an extra random distortion
            # (rotation + stronger colour, jointly) with probability RARE_EXTRA_PROB,
            # plus RandomErasing (occlusion) after ToTensor below.
            tile = ([transforms.RandomApply(
                [transforms.RandomRotation(20),
                 transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2)],
                p=RARE_EXTRA_PROB)] + tile)
    else:
        raise ValueError(augment)

    pipe = tile + [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    if augment == "rare" and is_rare:
        pipe.append(transforms.RandomErasing(p=RARE_EXTRA_PROB, scale=(0.02, 0.2)))
    elif augment == "strong":
        pipe.append(transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)))   # occlusion
    return transforms.Compose(pipe)

def val_transforms():
    return transforms.Compose([
        transforms.Resize(IMG_SIZE + 32),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def _apply_plain(ex, tfn):
    ex["image"] = [tfn(img.convert("RGB")) for img in ex["image"]]
    return ex

def _apply_rare(ex, rare_class):
    ex["image"] = [train_transforms("rare", int(lab) in rare_class)(img.convert("RGB"))
                   for img, lab in zip(ex["image"], ex["label"])]
    return ex

def get_dataloaders(augment, use_weighted_sampler=True):
    dataset = load_dataset("taufiktrf/AQUA20")
    split = dataset["train"].train_test_split(test_size=0.2, seed=42)
    train_split, val_split = split["train"], split["test"]
    test_split = dataset["test"]

    if use_weighted_sampler:
        train_labels = train_split["label"]
        class_counts = Counter(train_labels)
        rare_class = {int(l) for l, c in class_counts.items() if c < RARE_THRESHOLD}
        sample_weights = [1.0 / class_counts[l] for l in train_labels]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        shuffle = False
    else:
        sampler = None
        shuffle = True
        rare_class = set()

    if augment == "rare":
        train_split.set_transform(lambda ex: _apply_rare(ex, rare_class))
    else:
        tfn = train_transforms(augment, is_rare=False)
        train_split.set_transform(lambda ex: _apply_plain(ex, tfn))
    val_split.set_transform(lambda ex: _apply_plain(ex, val_transforms()))
    test_split.set_transform(lambda ex: _apply_plain(ex, val_transforms()))

    train_loader = DataLoader(train_split, batch_size=BATCH_SIZE, sampler=sampler,
                              shuffle=shuffle, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, val_loader, test_loader, rare_class

print("Train-time augmentation pipelines per policy:")
for a in AUGMENTATIONS:
    print(f"  {a}: {train_transforms(a)}")


## Model Builder — ResNet50

In [ ]:
def build_resnet50(num_classes=NUM_CLASSES):
    model = torchvision.models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    nn.init.kaiming_normal_(model.fc.weight, mode="fan_out", nonlinearity="relu")
    nn.init.zeros_(model.fc.bias)
    return model

def get_gradcam_target(model):
    return model.layer4[-1]

def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True


## Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, desc):
    model.train()
    total_loss = correct = total = 0
    pbar = tqdm(loader, desc=desc)
    for batch in pbar:
        images, labels = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({"loss": f"{total_loss/(pbar.n+1):.4f}", "acc": f"{correct/total:.4f}"})
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            images, batch_labels = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
            outputs = model(images)
            total_loss += criterion(outputs, batch_labels).item()
            all_preds.append(outputs.argmax(1).cpu())
            all_labels.append(batch_labels.cpu())
    p = torch.cat(all_preds).numpy()
    y = torch.cat(all_labels).numpy()
    acc = float((p == y).mean())
    macro_f1 = float(precision_recall_fscore_support(
        y, p, average="macro", zero_division=0)[2])
    return total_loss / len(loader), acc, macro_f1

def log_epoch(path, stage, epoch, train_loss, train_acc, val_loss, val_acc, val_f1,
              lr, elapsed, is_best):
    row = {
        "stage": stage, "epoch": epoch,
        "train_loss": round(train_loss, 5), "train_acc": round(train_acc, 5),
        "val_loss": round(val_loss, 5), "val_acc": round(val_acc, 5),
        "val_macro_f1": round(val_f1, 5),
        "lr": lr, "elapsed_sec": round(elapsed, 1), "is_best": bool(is_best),
    }
    pd.DataFrame([row]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

def selection_value(val_acc, val_f1):
    return val_f1 if SELECTION_METRIC == "macro_f1" else val_acc


## Training — two-stage, one run per augmentation policy

In [ ]:
criterion = nn.CrossEntropyLoss()
TRAINED = {}   # augment -> dict(paths, test_loader, epochs_stage2_run, stopped_early, rare_class)

def save_state(model, p, best_metric, val_acc, val_f1, stage, epoch):
    torch.save(model.state_dict(), p["checkpoint"])
    with open(p["state"], "w") as fh:
        json.dump({"selection_metric": SELECTION_METRIC, "best_metric": best_metric,
                   "best_val_acc": val_acc, "best_val_macro_f1": val_f1,
                   "stage": stage, "epoch": epoch, "seed": SEED}, fh)

def train_variant(augment):
    p = variant_paths(augment)
    print("\n" + "#" * 70)
    print(f"#  {augment.upper()} AUGMENTATION POLICY")
    print("#" * 70)
    train_loader, val_loader, test_loader, rare_class = get_dataloaders(augment, use_weighted_sampler=True)
    if augment == "rare":
        names = load_dataset_builder("taufiktrf/AQUA20").info.features["label"].names
        print(f"Rare classes (train < {RARE_THRESHOLD}): "
              + ", ".join(names[l] for l in sorted(rare_class)))
        print(f"  -> those samples also get the extra distortion w.p. {RARE_EXTRA_PROB}")

    model = build_resnet50(NUM_CLASSES).to(DEVICE)
    best_metric = 0.0
    start = time.time()

    # ---- Stage 1: head only ----
    freeze_backbone(model)
    classifier = model.fc
    optimizer = torch.optim.Adam(classifier.parameters(), lr=LR_HEAD)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.1)
    for epoch in range(1, EPOCHS_STAGE1 + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion,
            desc=f"[{augment}] Stage1 Epoch {epoch}/{EPOCHS_STAGE1}")
        val_loss, val_acc, val_f1 = validate(model, val_loader, criterion)
        current = selection_value(val_acc, val_f1)
        scheduler.step(current)
        print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")
        is_best = current > best_metric
        if is_best:
            best_metric = current
            save_state(model, p, best_metric, val_acc, val_f1, 1, epoch)
            print(f"  -> saved (best {SELECTION_METRIC}: {best_metric:.4f})")
        log_epoch(p["epoch_log"], 1, epoch, train_loss, train_acc, val_loss, val_acc, val_f1,
                  optimizer.param_groups[0]["lr"], time.time() - start, is_best)

    # ---- Stage 2: full fine-tune ----
    unfreeze_all(model)
    classifier_params_id = {id(pp) for pp in classifier.parameters()}
    backbone_params = [pp for pp in model.parameters() if id(pp) not in classifier_params_id]
    optimizer = torch.optim.Adam([
        {"params": backbone_params, "lr": LR_BACKBONE},
        {"params": classifier.parameters(), "lr": LR_HEAD},
    ])
    scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=5, factor=0.1)
    stopped_early = False
    epochs_run = EPOCHS_STAGE2
    for epoch in range(1, EPOCHS_STAGE2 + 1):
        elapsed = time.time() - start
        if elapsed > TIME_BUDGET_SEC:
            stopped_early = True
            print(f"\nTime budget ({TIME_BUDGET_SEC/3600:.1f} h) hit before stage-2 epoch {epoch} "
                  f"for [{augment}]. Stopping; best checkpoint already on disk.")
            epochs_run = epoch - 1
            break
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion,
            desc=f"[{augment}] Stage2 Epoch {epoch}/{EPOCHS_STAGE2}")
        val_loss, val_acc, val_f1 = validate(model, val_loader, criterion)
        current = selection_value(val_acc, val_f1)
        scheduler.step(current)
        print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")
        is_best = current > best_metric
        if is_best:
            best_metric = current
            save_state(model, p, best_metric, val_acc, val_f1, 2, epoch)
            print(f"  -> updated (best {SELECTION_METRIC}: {best_metric:.4f})")
        log_epoch(p["epoch_log"], 2, epoch, train_loss, train_acc, val_loss, val_acc, val_f1,
                  optimizer.param_groups[0]["lr"], time.time() - start, is_best)

    mins = (time.time() - start) / 60
    print(f"\n[{augment}] done in {mins:.1f} min. Best val {SELECTION_METRIC}: {best_metric:.4f}")
    TRAINED[augment] = {"paths": p, "test_loader": test_loader,
                        "epochs_stage2_run": epochs_run, "stopped_early": stopped_early}

for augment in AUGMENTATIONS:
    train_variant(augment)

print("\nAll augmentation policies trained.")


## Evaluation — metrics for every policy + comparison

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        images, batch_labels = batch["image"].to(DEVICE), batch["label"]
        probs = torch.softmax(model(images), dim=1)
        all_probs.append(probs.cpu())
        all_labels.append(batch_labels)
    return torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()

class_names = load_dataset_builder("taufiktrf/AQUA20").info.features["label"].names
print(f"Classes ({len(class_names)}): {class_names}\n")

EVAL = {}   # augment -> dict(labels, probs, preds, metrics, derived arrays)
for augment in AUGMENTATIONS:
    p = TRAINED[augment]["paths"]
    model = build_resnet50(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(p["checkpoint"], map_location=DEVICE, weights_only=True))
    labels, probs = evaluate(model, TRAINED[augment]["test_loader"])

    ranking = np.argsort(-probs, axis=1)
    preds = ranking[:, 0]
    top3_preds = ranking[:, :3]
    rows = np.arange(len(labels))
    confidence = probs[rows, ranking[:, 0]]
    true_prob = probs[rows, labels]
    correct = preds == labels

    top1 = accuracy_score(labels, preds)
    top3 = float(np.mean([labels[i] in top3_preds[i] for i in rows]))
    prec, rec, f1s, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    wf1 = float(precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)[2])

    EVAL[augment] = {"labels": labels, "probs": probs, "preds": preds,
                     "ranking": ranking, "top3_preds": top3_preds,
                     "confidence": confidence, "true_prob": true_prob, "correct": correct,
                     "top1": top1, "top3": top3, "precision": prec, "recall": rec,
                     "f1": f1s, "weighted_f1": wf1,
                     "n_errors": int(len(rows) - int(correct.sum()))}

    print("=" * 60)
    print(f"  {augment.upper()}  Top-1: {top1:.4f} | Top-3: {top3:.4f} | Macro-F1: {f1s:.4f}")
    print("=" * 60)


In [ ]:
# Compact per-policy comparison + bar chart
comp = []
for augment in AUGMENTATIONS:
    e = EVAL[augment]
    comp.append({"augmentation": augment,
                 "top1_accuracy": round(e["top1"], 4),
                 "top3_accuracy": round(e["top3"], 4),
                 "macro_precision": round(e["precision"], 4),
                 "macro_recall": round(e["recall"], 4),
                 "macro_f1": round(e["f1"], 4),
                 "weighted_f1": round(e["weighted_f1"], 4),
                 "n_errors": e["n_errors"],
                 "epochs_stage2_run": TRAINED[augment]["epochs_stage2_run"],
                 "stopped_early": TRAINED[augment]["stopped_early"]})
comp_df = pd.DataFrame(comp)
comp_df.to_csv("analysis/augmentation_comparison.csv", index=False)
print("Top-1 Accuracy vs Macro-F1 across augmentation policies:")
for _, r in comp_df.iterrows():
    print(f"  {r['augmentation']:8}  Top-1 {r['top1_accuracy']*100:5.2f}%   "
          f"Macro-F1 {r['macro_f1']:.4f}   errors {int(r['n_errors'])}")

plt.figure(figsize=(8, 6))
x = np.arange(len(comp_df))
w = 0.35
plt.bar(x - w/2, comp_df["top1_accuracy"] * 100, w, label="Top-1 accuracy (%)")
plt.bar(x + w/2, comp_df["macro_f1"] * 100, w, label="Macro F1 (%)")
plt.xticks(x, comp_df["augmentation"])
plt.ylabel("Percent")
plt.title("ResNet50 — data-augmentation comparison (AQUA20 test set)")
plt.legend()
plt.tight_layout()
plt.savefig("analysis/augmentation_comparison.png", dpi=150)
plt.show()
print("Saved analysis/augmentation_comparison.{csv,png}")


## Confusion matrices — one per policy

In [ ]:
for augment in AUGMENTATIONS:
    e = EVAL[augment]
    cmx = confusion_matrix(e["labels"], e["preds"])
    plt.figure(figsize=(12, 10))
    sns.heatmap(cmx, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix — ResNet50 [{augment}] (Top-1: {e['top1']*100:.2f}%)")
    plt.tight_layout()
    pth = variant_paths(augment)["confusion"]
    plt.savefig(pth, dpi=150)
    plt.close()
    print(f"{augment}: confusion matrix -> {pth}")


## Error analysis (per policy)

Writes small CSV/JSON artifacts to `analysis/<policy>/`. These download first and survive a
truncated pull, unlike the log they would otherwise only exist in.


In [ ]:
for augment in AUGMENTATIONS:
    A = variant_paths(augment)["analysis"]
    e = EVAL[augment]
    labels, preds = e["labels"], e["preds"]
    probs, ranking = e["probs"], e["ranking"]
    top3 = e["top3_preds"]
    confidence, true_prob, correct = e["confidence"], e["true_prob"], e["correct"]
    cmx = confusion_matrix(labels, preds)

    p_c, r_c, f_c, sup = precision_recall_fscore_support(
        labels, preds, labels=range(NUM_CLASSES), zero_division=0)
    per_class = []
    for c in range(NUM_CLASSES):
        m = labels == c
        hit, miss = m & correct, m & ~correct
        per_class.append({
            "class": class_names[c], "support": int(sup[c]),
            "precision": round(float(p_c[c]), 4), "recall": round(float(r_c[c]), 4),
            "f1": round(float(f_c[c]), 4),
            "top3_recall": round(float(np.mean([labels[i] in top3[i] for i in np.flatnonzero(m)])), 4)
                           if m.any() else 0.0,
            "mean_conf_correct": round(float(confidence[hit].mean()), 4) if hit.any() else None,
            "mean_conf_wrong": round(float(confidence[miss].mean()), 4) if miss.any() else None,
            "mean_true_prob_when_wrong": round(float(true_prob[miss].mean()), 4) if miss.any() else None,
        })
    per_class_df = pd.DataFrame(per_class).sort_values("f1")
    per_class_df.to_csv(f"{A}/per_class.csv", index=False)

    pd.DataFrame(cmx, index=class_names, columns=class_names).to_csv(f"{A}/confusion_matrix.csv")

    pairs = [{"true": class_names[t], "predicted": class_names[p], "count": int(cmx[t, p]),
              "pct_of_true_class": round(100.0 * cmx[t, p] / max(cmx[t].sum(), 1), 2)}
             for t in range(NUM_CLASSES) for p in range(NUM_CLASSES)
             if t != p and cmx[t, p] > 0]
    pairs_df = pd.DataFrame(pairs).sort_values("count", ascending=False)
    pairs_df.to_csv(f"{A}/confusion_pairs.csv", index=False)

    wrong_idx = np.flatnonzero(~correct)
    mistakes = pd.DataFrame({
        "test_index": wrong_idx,
        "true": [class_names[c] for c in labels[wrong_idx]],
        "predicted": [class_names[c] for c in preds[wrong_idx]],
        "confidence": np.round(confidence[wrong_idx], 4),
        "true_class_prob": np.round(true_prob[wrong_idx], 4),
        "true_rank": [int(np.where(ranking[i] == labels[i])[0][0]) + 1 for i in wrong_idx],
    }).sort_values("confidence", ascending=False)
    mistakes.to_csv(f"{A}/confident_mistakes.csv", index=False)

    p = variant_paths(augment)
    st = json.load(open(p["state"])) if os.path.exists(p["state"]) else {}
    tmp = build_resnet50(NUM_CLASSES)
    n_params = int(sum(pp.numel() for pp in tmp.parameters()))
    summary = {
        "model": MODEL_NAME, "augmentation": augment,
        "n_test": int(len(labels)), "top1_accuracy": round(float(e["top1"]), 4),
        "top3_accuracy": round(float(e["top3"]), 4),
        "macro_precision": round(float(e["precision"]), 4),
        "macro_recall": round(float(e["recall"]), 4),
        "macro_f1": round(float(e["f1"]), 4),
        "weighted_f1": round(float(e["weighted_f1"]), 4),
        "selection_metric": SELECTION_METRIC,
        "best_val_metric": round(float(st.get("best_metric", 0.0)), 4),
        "best_val_acc": round(float(st["best_val_acc"]), 4) if st.get("best_val_acc") is not None else None,
        "best_val_macro_f1": round(float(st["best_val_macro_f1"]), 4) if st.get("best_val_macro_f1") is not None else None,
        "best_epoch": st.get("epoch"), "best_stage": st.get("stage"),
        "n_errors": int(e["n_errors"]),
        "mean_confidence_correct": round(float(confidence[correct].mean()), 4),
        "mean_confidence_wrong": round(float(confidence[~correct].mean()), 4),
        "epochs_stage2_run": int(TRAINED[augment]["epochs_stage2_run"]),
        "stopped_early": bool(TRAINED[augment]["stopped_early"]),
        "weighted_sampler": True, "seed": SEED,
        "rare_threshold": RARE_THRESHOLD, "rare_extra_prob": RARE_EXTRA_PROB,
        "n_params": n_params,
    }
    json.dump(summary, open(f"{A}/summary.json", "w"), indent=2)
    np.save(f"{A}/probabilities.npy", probs.astype(np.float32))

    print(f"\n=== [{augment}] {int(e['n_errors'])} errors ({100*e['n_errors']/len(labels):.1f}%) "
          f"| mean conf correct {summary['mean_confidence_correct']:.4f} wrong "
          f"{summary['mean_confidence_wrong']:.4f}")
    print(per_class_df.head(5).to_string(index=False))
    print(pairs_df.head(8).to_string(index=False))
print("\nError analysis written for all policies.")


## Grad-CAM Explainability (per policy)

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = self.activations = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, m, inp, out): self.activations = out.detach()
    def _bwd_hook(self, m, gi, go): self.gradients = go[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        with torch.enable_grad():
            input_tensor = input_tensor.unsqueeze(0).requires_grad_(True)
            output = self.model(input_tensor)
            if class_idx is None:
                class_idx = output.argmax(dim=1).item()
            one_hot = torch.zeros_like(output)
            one_hot[0, class_idx] = 1
            self.model.zero_grad()
            output.backward(gradient=one_hot)
        with torch.no_grad():
            grads, acts = self.gradients, self.activations
            weights = grads.mean(dim=(2, 3), keepdim=True)
            cam = torch.relu((weights * acts).sum(dim=1)).squeeze(0).cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        h, w = input_tensor.shape[2:]
        cam = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)
        return cam, class_idx

def unnormalize(tensor):
    img = tensor.cpu().clone()
    for t, m, s in zip(img, IMAGENET_MEAN, IMAGENET_STD):
        t.mul_(s).add_(m)
    return img.permute(1, 2, 0).numpy()

def overlay_heatmap(img, cam, alpha=0.5):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return (alpha * heatmap + (1 - alpha) * 255 * img).astype(np.uint8)


## Grad-CAM visualisations — one figure-set per policy

In [ ]:
GRADCAM_N_WRONG = 20
GRADCAM_N_RIGHT = 20

def select_gradcam_indices(labels, preds, confidence, n_wrong, n_right):
    def spread(pool, n):
        if len(pool) == 0:
            return []
        by_class = {}
        for idx in pool:
            by_class.setdefault(int(labels[idx]), []).append(int(idx))
        for c in by_class:
            by_class[c].sort(key=lambda i: -confidence[i])
        picked, classes = [], sorted(by_class)
        while len(picked) < n and any(by_class[c] for c in classes):
            for c in classes:
                if by_class[c] and len(picked) < n:
                    picked.append(by_class[c].pop(0))
        return picked

    wrong = spread(np.flatnonzero(labels != preds), n_wrong)
    right = spread(np.flatnonzero(labels == preds), n_right)
    return wrong, right

for augment in AUGMENTATIONS:
    e = EVAL[augment]
    labels, preds, confidence = e["labels"], e["preds"], e["confidence"]
    gdir = variant_paths(augment)["gradcam"]

    model = build_resnet50(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(variant_paths(augment)["checkpoint"],
                                     map_location=DEVICE, weights_only=True))
    target_layer = get_gradcam_target(model)
    gradcam = GradCAM(model, target_layer)

    wrong_sel, right_sel = select_gradcam_indices(labels, preds, confidence, GRADCAM_N_WRONG, GRADCAM_N_RIGHT)
    selected = wrong_sel + right_sel
    n_classes = len({int(labels[i]) for i in selected})
    print(f"[{augment}] {len(wrong_sel)} misclassified + {len(right_sel)} correct, "
          f"{n_classes} classes -> {gdir}")

    gc_loader = DataLoader(Subset(TRAINED[augment]["test_loader"].dataset, selected),
                           batch_size=1, shuffle=False, num_workers=NUM_WORKERS)
    manifest = []
    for ii, batch in enumerate(gc_loader):
        ds_idx = selected[ii]
        label, pred = int(labels[ds_idx]), int(preds[ds_idx])
        img_tensor = batch["image"].to(DEVICE)
        cam, _ = gradcam.generate(img_tensor.squeeze(0), class_idx=pred)
        img_np = np.clip(unnormalize(img_tensor.squeeze(0)), 0, 1)
        overlay = overlay_heatmap(img_np, cam)
        is_ok = label == pred
        conf = float(confidence[ds_idx])
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(img_np); axes[0].set_title(f"True: {class_names[label]}"); axes[0].axis("off")
        axes[1].imshow(cam, cmap="jet", vmin=0, vmax=1); axes[1].set_title("Grad-CAM"); axes[1].axis("off")
        axes[2].imshow(overlay); axes[2].axis("off")
        axes[2].set_title(f"Pred: {class_names[pred]} ({conf:.2f})" + ("" if is_ok else "  X"))
        plt.tight_layout()
        tag = "zz_ok" if is_ok else "err"
        fname = f"{tag}_{ii:03d}_{class_names[label]}-as-{class_names[pred]}.png"
        plt.savefig(f"{gdir}/{fname}", dpi=150); plt.close(fig)
        manifest.append({"file": fname, "test_index": ds_idx, "true": class_names[label],
                         "predicted": class_names[pred], "confidence": round(conf, 4),
                         "correct": bool(is_ok)})
    pd.DataFrame(manifest).to_csv(f"{variant_paths(augment)['analysis']}/gradcam_manifest.csv", index=False)
    print(f"  saved {len(selected)} images")

print("\nGrad-CAM complete for all policies.")


## Outputs

In [ ]:
total = 0
print("Kernel output (/kaggle/working):")
for root, _, files in os.walk("."):
    if any(pt in root for pt in (".git", "__pycache__", ".ipynb_checkpoints")):
        continue
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        total += size
        print(f"  {os.path.relpath(path):62} {size/1e6:8.2f} MB")
print(f"\nTotal: {total/1e6:.1f} MB")
print("\nPull with:  kaggle kernels output farhantahsinkhan/<slug> -p <dir>")
print("Downloads are alphabetical and can truncate — analysis/ arrives first, weights last.")
